# Matching Embeddings — TF two-tower + FAISS (Food.com interactions)

Recsys-style two-tower on Food.com user<->recipe interactions
(`RAW_interactions.csv`, 1.13M rows, 2.2e-5 density). Positive = rating >= 4
implicit feedback, negatives sampled uniformly from recipes. Serves
`app/embedding_engine.py` `/api/v1/embeddings/match`. Artifacts: the
`matching_embeddings` ONNX user-tower and the `matching_items.faiss` index plus
ID maps for serving-side lookup.

In [1]:
import importlib.util
import os
import pathlib
import sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
sys.path.insert(0, str(ai))         # for the training.* namespace package
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'faiss-cpu'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

# -- shared bootstrap: seeds RNGs, caps CPU threads, resolves data/output roots ----
try:
    from training.bootstrap import *  # noqa: F403
    CFG = init()
except Exception as _boot_err:  # bootstrap is an upgrade, never brick a notebook
    print('[bootstrap] unavailable:', repr(_boot_err))
    CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


2026-08-04 23:06:52.419732: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-04 23:06:52.530744: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-04 23:06:54.954421: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TF 2.20.0 | GPU: False | scale: demo


2026-08-04 23:06:57.762069: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Load interactions; implicit feedback = rating >= 4
import os
import numpy as np
from buddy_data import food_com_interactions

N = {'smoke': 60_000, 'demo': 300_000, 'full': 1_000_000}[SCALE]
it = food_com_interactions()
it = it[it['rating'] >= 4]
rng = np.random.default_rng(42)
it = it.sample(n=min(N, len(it)), random_state=42)
print('sampled interactions:', it.shape)

u_cat = it['user_id'].astype('category')
i_cat = it['recipe_id'].astype('category')
it['u'] = u_cat.cat.codes.values
it['i'] = i_cat.cat.codes.values
n_users, n_items = u_cat.cat.categories.size, i_cat.cat.categories.size
print('users:', n_users, 'items:', n_items)

sampled interactions: (300000, 5)
users: 76337 items: 118735


In [3]:
# User-level split: hold out ~10% of users entirely so the same
# user (and their items) never straddles train and validation.
pos_by_user = it.groupby('u')['i'].apply(set).to_dict()
all_users = np.unique(it["u"].to_numpy())
val_uids = set(rng.permutation(all_users)[:max(1, int(0.1 * len(all_users)))].tolist())
is_val = it["u"].isin(val_uids).to_numpy()
train_pos, val_pos = it[~is_val], it[is_val]


def sample_negatives(u_arr, filter_known):
    """1 uniform-random negative per positive.

    filter_known resamples (bounded) negatives that collide with the
    user's known positives — used for validation so no "negative" is
    secretly a positive.
    """
    neg = rng.integers(0, n_items, size=len(u_arr))
    if not filter_known:
        return neg
    for _ in range(10):
        collide = np.array([neg[j] in pos_by_user.get(int(u), ())
                            for j, u in enumerate(u_arr)])
        if not collide.any():
            break
        neg[collide] = rng.integers(0, n_items, size=int(collide.sum()))
    return neg


def build_pairs(pos_df, filter_negatives):
    u = pos_df["u"].to_numpy(np.int64)
    i = pos_df["i"].to_numpy(np.int64)
    neg = sample_negatives(u, filter_negatives).astype(np.int64)
    if filter_negatives:  # drop any residual collisions (rare)
        ok = np.array([neg[j] not in pos_by_user.get(int(u), ())
                       for j, u in enumerate(u)])
        u, i, neg = u[ok], i[ok], neg[ok]
    users = np.concatenate([u, u])
    items = np.concatenate([i, neg])
    labels = np.concatenate([np.ones(len(u)), np.zeros(len(u))]).astype(np.float32)
    order = rng.permutation(len(users))
    return users[order], items[order], labels[order]


Xu_tr, Xi_tr, Y_tr = build_pairs(train_pos, filter_negatives=False)
Xu_va, Xi_va, Y_va = build_pairs(val_pos, filter_negatives=True)
print('train pairs:', len(Y_tr), '| val pairs:', len(Y_va),
      f'({len(val_uids)} held-out users)')

train pairs: 540000 | val pairs: 60000


In [4]:
# Two-tower with logit output (recompile for BCE)
import tensorflow as tf
from tf_utils import build_two_tower

m = build_two_tower(embed_dim=64, n_users=n_users, n_items=n_items)
m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
m.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 64)     │  4,885,568 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 64)     │  7,599,040 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 64)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 64)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,484,608 (47.62 MB)

 Trainable params: 12,484,608 (47.62 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Train (embedding lookups — fast on CPU)
import time
EPOCHS = {'smoke': 1, 'demo': 5, 'full': 10}[SCALE]
t0 = time.time()
m.fit([Xu_tr, Xi_tr], Y_tr, epochs=EPOCHS, batch_size=1024,
      validation_data=([Xu_va, Xi_va], Y_va), verbose=1)
print(f'train {time.time()-t0:.0f}s')

Epoch 1/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 63s 116ms/step - accuracy: 0.5001 - loss: 5.3935 - val_accuracy: 0.4992 - val_loss: 5.3386
Epoch 2/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 63s 120ms/step - accuracy: 0.5001 - loss: 4.4256 - val_accuracy: 0.4992 - val_loss: 5.2952
Epoch 3/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 62s 118ms/step - accuracy: 0.5001 - loss: 4.0415 - val_accuracy: 0.4992 - val_loss: 5.2481
Epoch 4/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 64s 122ms/step - accuracy: 0.5001 - loss: 3.8025 - val_accuracy: 0.4992 - val_loss: 5.2025
Epoch 5/5
528/528 ━━━━━━━━━━━━━━━━━━━━ 67s 127ms/step - accuracy: 0.5001 - loss: 3.6153 - val_accuracy: 0.4992 - val_loss: 5.1693
train 320s


In [6]:
# Ranking eval: HR@10 / MRR over 100 random distractors for held-out users
import numpy as np

user_emb = m.layers[2].get_weights()[0]      # (n_users, 64)
item_emb = m.layers[3].get_weights()[0]      # (n_items, 64)
pos_pairs = np.unique(np.stack([Xu_va, Xi_va], axis=1), axis=0)

rng = np.random.default_rng(0)
hits, rr, n_eval = 0, 0.0, 0
for u, i in pos_pairs[:2000]:
    cand = np.concatenate([[i], rng.integers(0, n_items, size=99)])
    scores = item_emb[cand] @ user_emb[u]
    order = np.argsort(scores)[::-1]               # rank 0 = best
    rank = int(np.where(order == 0)[0][0]) + 1
    hits += int(rank <= 10)
    rr += 1.0 / rank
    n_eval += 1
print(f'HR@10={hits/n_eval:.3f}  MRR={rr/n_eval:.3f}  (n={n_eval})')

HR@10=0.096  MRR=0.052  (n=2000)


### Export contract (consumed by the AI service)

The cells below write `../models/matching_embeddings.onnx` and its dynamic-INT8 quantized copy
`matching_embeddings_int8.onnx`. `app/ml/serving.py::load_preferred('matching_embeddings')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [7]:
# Build + persist FAISS item index and ID maps for serving
import faiss

item_emb = item_emb.astype('float32')
faiss.normalize_L2(item_emb)
index = faiss.IndexFlatIP(item_emb.shape[1])
index.add(item_emb)
faiss.write_index(index, '../models/matching_items.faiss')
np.save('../models/matching_users.npy', u_cat.cat.categories.to_numpy())
np.save('../models/matching_recipes.npy', i_cat.cat.categories.to_numpy())
print('FAISS index items:', index.ntotal)

# quick sanity: top-5 for a random user
u = int(pos_pairs[0][0])
print('top5 for user', u_cat.cat.categories[u], '->',
      [int(x) for x in index.search(user_emb[u:u+1].astype('float32'), 5)[1][0]])

FAISS index items: 118735
top5 for user 1533 -> [39649, 11507, 8972, 6076, 12714]


In [8]:
# Export the user tower as ONNX (+ INT8) for online matching
from pathlib import Path
import tensorflow as tf
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

emb_dim = m.layers[2].output.shape[-1]
tinp = tf.keras.Input(shape=(1,), dtype='int64')
tx = tf.keras.layers.Embedding(n_users, emb_dim,
                               weights=[m.layers[2].get_weights()[0]])(tinp)
tower = tf.keras.Model(tinp, tf.keras.layers.Flatten()(tx))
onnx = export_keras_onnx(tower, Path('../models'), 'matching_embeddings', '1.0.0',
                         input_signature=[tf.TensorSpec((None, 1), tf.int64, name='user_id')])
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'matching_embeddings', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'hr@10': round(hits / n_eval, 4), 'mrr': round(rr / n_eval, 4)}})
print('exported', q)

I0000 00:00:1785874346.749799 1056498 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785874346.750012 1056498 single_machine.cc:376] Starting new session
I0000 00:00:1785874347.395209 1056498 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785874347.395435 1056498 single_machine.cc:376] Starting new session


{
  "name": "matching_embeddings",
  "version": "1.0.0",
  "artifact_path": "../models/matching_embeddings-1.0.0_int8.onnx",
  "framework": "tensorflow",
  "metrics": {
    "hr@10": 0.0955,
    "mrr": 0.052
  }
}
exported ../models/matching_embeddings-1.0.0_int8.onnx
